In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score
from dataclasses import dataclass
from typing import List

# Config
DATA_NPZ_PATH = "../datasets/soil_spi_preprocessed.npz"

OUT_DIR = "./outputs_globalsoil_accuracy_multisize"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
SAMPLE_SEED = 123
N_GLOBAL_SAMPLES = 50

DATA_SIZES = [500, 1000, "full"]

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.banzhaf import exact_banzhaf_index

# Data structure
@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]

    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

# Global accuracy utility
class GlobalAccuracyUtility:
    def __init__(self, model, X_eval, y_eval, baseline_full, feature_names):
        self.model = model
        self.X_eval = np.asarray(X_eval, dtype=float)
        self.y_eval = np.asarray(y_eval, dtype=int).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_names = feature_names

        self.n_features = self.X_eval.shape[1]

        self.data = FeaturePlayers(
            indices=list(range(self.n_features)),
            data_names=[feature_names[j] for j in range(self.n_features)]
        )

    def _X_for_coalition(self, S):
        X_masked = np.tile(self.baseline_full, (self.X_eval.shape[0], 1))

        for j in S:
            j = int(j)
            X_masked[:, j] = self.X_eval[:, j]

        return X_masked

    def __call__(self, S):
        X_in = self._X_for_coalition(S)
        preds = self.model.predict(X_in)
        return float(accuracy_score(self.y_eval, preds))

# Methods / models
METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index),
]

MODEL_NAMES = ["logistic", "svm", "random_forest"]

# Helpers
def load_data(npz_path):
    d = np.load(npz_path, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Original train size: {X_train.shape[0]}")
    print(f"Test size: {X_test.shape[0]}")
    print(f"Number of features: {X_train.shape[1]}")
    print("Feature order:")
    for i, name in enumerate(feature_names):
        print(f"  {i}: {name}")

    return X_train, y_train, X_test, y_test, feature_names

def get_train_subset(X_train_orig, y_train_orig, size_tag):
    if size_tag == "full":
        return X_train_orig, y_train_orig

    rng = np.random.default_rng(SAMPLE_SEED)
    idx = rng.choice(len(X_train_orig), size=int(size_tag), replace=False)

    return X_train_orig[idx], y_train_orig[idx]

# Main
def main():
    X_train_orig, y_train_orig, X_test, y_test, feature_names = load_data(DATA_NPZ_PATH)

    rng_eval = np.random.default_rng(SAMPLE_SEED)
    eval_idx = rng_eval.choice(
        len(X_test),
        size=min(N_GLOBAL_SAMPLES, len(X_test)),
        replace=False
    )

    X_eval = X_test[eval_idx]
    y_eval = y_test[eval_idx]

    print(f"\nUsing {len(eval_idx)} fixed global evaluation samples")

    for size_tag in DATA_SIZES:
        print("\n" + "=" * 80)
        print(f"GLOBAL ACCURACY UTILITY | Train size = {size_tag}")
        print("=" * 80)

        X_train, y_train = get_train_subset(X_train_orig, y_train_orig, size_tag)

        baseline_full = np.mean(X_train, axis=0)

        records = []

        for model_name in MODEL_NAMES:
            print(f"\nTraining model: {model_name}")

            model = get_model(model_name, random_state=RANDOM_STATE)
            model.fit(X_train, y_train)

            train_acc = accuracy_score(y_train, model.predict(X_train))
            test_acc = accuracy_score(y_test, model.predict(X_test))
            eval_acc = accuracy_score(y_eval, model.predict(X_eval))

            print(
                f"{model_name} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Full Test Acc: {test_acc:.4f} | "
                f"Eval Acc: {eval_acc:.4f}"
            )

            util = GlobalAccuracyUtility(
                model=model,
                X_eval=X_eval,
                y_eval=y_eval,
                baseline_full=baseline_full,
                feature_names=feature_names
            )

            for method_name, method_fn in METHODS:
                print(f"  Computing {method_name}")

                try:
                    res = method_fn(util)
                    alloc = getattr(res, "values", res)
                    alloc = np.asarray(alloc, dtype=float).ravel()

                    records.append({
                        "setup": "global",
                        "utility": "accuracy",
                        "dataset_size": str(size_tag),
                        "dataset_train_n": int(X_train.shape[0]),
                        "model": model_name,
                        "method": method_name,
                        "train_accuracy": float(train_acc),
                        "test_accuracy": float(test_acc),
                        "eval_accuracy": float(eval_acc),
                        "alloc_global": json.dumps(alloc.tolist()),
                        "eval_indices": json.dumps(eval_idx.tolist()),
                        "n_eval_samples": int(len(eval_idx))
                    })

                except Exception as e:
                    print(f"  ERROR: {method_name}, model={model_name}: {e}")

        df = pd.DataFrame(records)

        out_file = os.path.join(
            OUT_DIR,
            f"global_accuracy_results_size_{size_tag}.csv"
        )

        df.to_csv(out_file, index=False)

        print(f"\nSaved: {out_file}")
        print(df[[
            "utility",
            "dataset_size",
            "dataset_train_n",
            "model",
            "method",
            "train_accuracy",
            "test_accuracy",
            "eval_accuracy"
        ]])

if __name__ == "__main__":
    main()

Original train size: 4000
Test size: 1000
Number of features: 7
Feature order:
  0: Organic Carbon
  1: Nitrogen
  2: Phosphorus
  3: Potassium
  4: pH
  5: Zinc
  6: Electrical Conductivity

Using 50 fixed global evaluation samples

GLOBAL ACCURACY UTILITY | Train size = 500

Training model: logistic
logistic | Train Acc: 0.8440 | Full Test Acc: 0.8420 | Eval Acc: 0.8400
  Computing shapley
  Computing least_core
  Computing nucleolus
  Computing gately
  Computing banzhaf

Training model: svm
svm | Train Acc: 0.9180 | Full Test Acc: 0.8760 | Eval Acc: 0.9200
  Computing shapley
  Computing least_core
  Computing nucleolus
  Computing gately
  Computing banzhaf

Training model: random_forest
random_forest | Train Acc: 1.0000 | Full Test Acc: 0.8670 | Eval Acc: 0.8800
  Computing shapley
  Computing least_core
  Computing nucleolus
  Computing gately
  Computing banzhaf

Saved: ./outputs_globalsoil_accuracy_multisize/global_accuracy_results_size_500.csv
     utility dataset_size  datas

In [2]:
import os
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus

from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score
from numpy.linalg import norm

sys.path.append(os.path.abspath(".."))

from models.train_model import get_model

# Config
DATA_NPZ_PATH = "../datasets/soil_spi_preprocessed.npz"

GLOBAL_RESULTS_DIR = "./outputs_globalsoil_accuracy_multisize"
OUT_DIR = Path("./outputs_globalsoil_accuracy_gt_quantus")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_SIZES = ["500", "1000", "full"]
RANDOM_STATE = 42
SAMPLE_SEED = 123
AOPC_TOPK_LIST = [1,2,3,4,5,6,7]

# Ground truth
GT_RAW = np.array([0.064, 0.401, 0.047, 0.138, 0.138, 0.060, 0.153])

FEATURE_NAMES = [
    "Organic Carbon","Nitrogen","Phosphorus",
    "Potassium","pH","Zinc","Electrical Conductivity"
]

def normalize_abs(v):
    v = np.abs(v)
    return v / (np.sum(v) + 1e-8)

GT = normalize_abs(GT_RAW)

# Model wrapper
class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        x_np = x.detach().cpu().numpy()

        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np).reshape(-1,1)

        return torch.from_numpy(out).float()

# Data
def load_data():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)

    return (
        d["X_train"].astype(float),
        d["y_train"].astype(int).ravel(),
        d["X_test"].astype(float),
        d["y_test"].astype(int).ravel()
    )

def get_subset(X, y, size):
    if size == "full":
        return X, y

    rng = np.random.default_rng(SAMPLE_SEED)
    idx = rng.choice(len(X), size=int(size), replace=False)
    return X[idx], y[idx]

# Perturbation
def perturb_tabular(arr, indices, baseline):
    arr = arr.copy()
    base = baseline.flatten()

    for b in range(arr.shape[0]):
        idxs = indices[b] if isinstance(indices[b], list) else [indices[b]]
        for i in idxs:
            arr[b,0,i] = base[i]
    return arr

# Metrics
def get_quantus_metrics(baseline):
    base_3d = baseline.reshape(1,1,-1)

    return {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=5,
            subset_size=1,
            perturb_func=perturb_tabular,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),
        "selectivity": quantus.Selectivity(
            perturb_func=perturb_tabular,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),
        "monotonicity": quantus.Monotonicity(
            perturb_func=perturb_tabular,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),
        "pixel_flipping": quantus.PixelFlipping(
            perturb_func=perturb_tabular,
            perturb_func_kwargs={"baseline": base_3d},
            return_aggregate=True
        ),
        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True)
    }

# Aopc
def compute_aopc(model, X, y, attr, baseline):
    probs = model.predict_proba(X)
    base_scores = probs[np.arange(len(y)), y]

    drops = []
    for k in AOPC_TOPK_LIST:
        Xp = X.copy()

        for i in range(len(X)):
            topk = np.argsort(-np.abs(attr[i]))[:k]
            Xp[i, topk] = baseline[topk]

        new_probs = model.predict_proba(Xp)
        new_scores = new_probs[np.arange(len(y)), y]

        drops.append(np.mean(base_scores - new_scores))

    return np.mean(drops)

# Main
def main():
    X_train_full, y_train_full, X_test, y_test = load_data()

    all_rows = []

    for size in DATA_SIZES:
        print("\n" + "="*80)
        print(f"SIZE: {size}")
        print("="*80)

        csv_path = os.path.join(
            GLOBAL_RESULTS_DIR,
            f"global_accuracy_results_size_{size}.csv"
        )

        if not os.path.exists(csv_path):
            print("Missing:", csv_path)
            continue

        df = pd.read_csv(csv_path)

        X_train, y_train = get_subset(X_train_full, y_train_full, size)
        baseline = np.mean(X_train, axis=0)

        metrics = get_quantus_metrics(baseline)

        eval_idx = json.loads(df.iloc[0]["eval_indices"])
        X_eval = X_test[eval_idx]
        y_eval = y_test[eval_idx]

        x_batch = X_eval[:,None,:]

        for model_name in df["model"].unique():

            model = get_model(model_name, random_state=RANDOM_STATE)
            model.fit(X_train, y_train)

            torch_model = SklearnTorchWrapper(model)

            for method in df["method"].unique():

                row = df[(df["model"]==model_name) & (df["method"]==method)].iloc[0]

                phi_raw = np.array(json.loads(row["alloc_global"]))
                phi = normalize_abs(phi_raw)

                # Repeat for batch
                a_batch = np.tile(phi, (len(X_eval),1))
                a_batch = a_batch[:,None,:]

                spearman = spearmanr(phi, GT).correlation
                cosine = np.dot(phi,GT)/(norm(phi)*norm(GT)+1e-8)
                rmse = np.sqrt(mean_squared_error(GT, phi))

                q_scores = {}

                for name, fn in metrics.items():
                    try:
                        scores = fn(
                            model=torch_model,
                            x_batch=x_batch,
                            y_batch=y_eval,
                            a_batch=a_batch,
                            channel_first=True
                        )
                        q_scores[name] = np.nanmean(scores)
                    except:
                        q_scores[name] = np.nan

                try:
                    aopc = compute_aopc(model, X_eval, y_eval, a_batch.squeeze(), baseline)
                except:
                    aopc = np.nan

                result = {
                    "size": size,
                    "model": model_name,
                    "method": method,
                    "spearman": spearman,
                    "cosine": cosine,
                    "rmse": rmse,
                    "aopc": aopc,
                    **q_scores
                }

                all_rows.append(result)

                print(
                    f"{model_name:15} {method:12} | "
                    f"Sp={spearman:.3f} | AOPC={aopc:.3f}"
                )

    df_out = pd.DataFrame(all_rows)
    out_path = OUT_DIR / "final_accuracy_gt_quantus.csv"
    df_out.to_csv(out_path, index=False)

    print("\nSaved to:", out_path)
    print(df_out.head())

if __name__ == "__main__":
    main()


SIZE: 500
logistic        shapley      | Sp=0.739 | AOPC=0.364
logistic        least_core   | Sp=0.847 | AOPC=0.373
logistic        nucleolus    | Sp=0.847 | AOPC=0.373
logistic        gately       | Sp=0.450 | AOPC=0.344
logistic        banzhaf      | Sp=0.847 | AOPC=0.373
svm             shapley      | Sp=0.793 | AOPC=0.425
svm             least_core   | Sp=0.667 | AOPC=0.416
svm             nucleolus    | Sp=0.667 | AOPC=0.416
svm             gately       | Sp=0.378 | AOPC=0.391
svm             banzhaf      | Sp=0.649 | AOPC=0.415
random_forest   shapley      | Sp=0.847 | AOPC=0.309
random_forest   least_core   | Sp=0.847 | AOPC=0.309
random_forest   nucleolus    | Sp=0.757 | AOPC=0.307
random_forest   gately       | Sp=0.818 | AOPC=0.301
random_forest   banzhaf      | Sp=0.739 | AOPC=0.309

SIZE: 1000
logistic        shapley      | Sp=0.775 | AOPC=0.375
logistic        least_core   | Sp=0.667 | AOPC=0.372
logistic        nucleolus    | Sp=0.667 | AOPC=0.372
logistic        gately 

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score
from dataclasses import dataclass
from typing import List

# Config
DATA_NPZ_PATH = "../datasets/soil_spi_preprocessed.npz"

OUT_DIR = "./outputs_globalsoil_accuracy_multisize"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
SAMPLE_SEED = 123
N_GLOBAL_SAMPLES = 50

DATA_SIZES = [500, 1000, "full"]

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.banzhaf import exact_banzhaf_index

# Data structure
@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]

    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

# Global accuracy utility
class GlobalAccuracyUtility:
    def __init__(self, model, X_eval, y_eval, baseline_full, feature_names):
        self.model = model
        self.X_eval = np.asarray(X_eval, dtype=float)
        self.y_eval = np.asarray(y_eval, dtype=int).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_names = feature_names

        self.n_features = self.X_eval.shape[1]

        self.data = FeaturePlayers(
            indices=list(range(self.n_features)),
            data_names=[feature_names[j] for j in range(self.n_features)]
        )

    def _X_for_coalition(self, S):
        X_masked = np.tile(self.baseline_full, (self.X_eval.shape[0], 1))

        for j in S:
            j = int(j)
            X_masked[:, j] = self.X_eval[:, j]

        return X_masked

    def __call__(self, S):
        X_in = self._X_for_coalition(S)
        preds = self.model.predict(X_in)
        return float(accuracy_score(self.y_eval, preds))

# Methods / models
METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index),
]

MODEL_NAMES = ["logistic", "svm", "random_forest"]

# Helpers
def load_data(npz_path):
    d = np.load(npz_path, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Original train size: {X_train.shape[0]}")
    print(f"Test size: {X_test.shape[0]}")
    print(f"Number of features: {X_train.shape[1]}")
    print("Feature order:")
    for i, name in enumerate(feature_names):
        print(f"  {i}: {name}")

    return X_train, y_train, X_test, y_test, feature_names

def get_train_subset(X_train_orig, y_train_orig, size_tag):
    if size_tag == "full":
        return X_train_orig, y_train_orig

    rng = np.random.default_rng(SAMPLE_SEED)
    idx = rng.choice(len(X_train_orig), size=int(size_tag), replace=False)

    return X_train_orig[idx], y_train_orig[idx]

# Main
def main():
    X_train_orig, y_train_orig, X_test, y_test, feature_names = load_data(DATA_NPZ_PATH)

    rng_eval = np.random.default_rng(SAMPLE_SEED)
    eval_idx = rng_eval.choice(
        len(X_test),
        size=min(N_GLOBAL_SAMPLES, len(X_test)),
        replace=False
    )

    X_eval = X_test[eval_idx]
    y_eval = y_test[eval_idx]

    print(f"\nUsing {len(eval_idx)} fixed global evaluation samples")

    for size_tag in DATA_SIZES:
        print("\n" + "=" * 80)
        print(f"GLOBAL ACCURACY UTILITY | Train size = {size_tag}")
        print("=" * 80)

        X_train, y_train = get_train_subset(X_train_orig, y_train_orig, size_tag)

        baseline_full = np.mean(X_train, axis=0)

        records = []

        for model_name in MODEL_NAMES:
            print(f"\nTraining model: {model_name}")

            model = get_model(model_name, random_state=RANDOM_STATE)
            model.fit(X_train, y_train)

            train_acc = accuracy_score(y_train, model.predict(X_train))
            test_acc = accuracy_score(y_test, model.predict(X_test))
            eval_acc = accuracy_score(y_eval, model.predict(X_eval))

            print(
                f"{model_name} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Full Test Acc: {test_acc:.4f} | "
                f"Eval Acc: {eval_acc:.4f}"
            )

            util = GlobalAccuracyUtility(
                model=model,
                X_eval=X_eval,
                y_eval=y_eval,
                baseline_full=baseline_full,
                feature_names=feature_names
            )

            for method_name, method_fn in METHODS:
                print(f"  Computing {method_name}")

                try:
                    res = method_fn(util)
                    alloc = getattr(res, "values", res)
                    alloc = np.asarray(alloc, dtype=float).ravel()

                    records.append({
                        "setup": "global",
                        "utility": "accuracy",
                        "dataset_size": str(size_tag),
                        "dataset_train_n": int(X_train.shape[0]),
                        "model": model_name,
                        "method": method_name,
                        "train_accuracy": float(train_acc),
                        "test_accuracy": float(test_acc),
                        "eval_accuracy": float(eval_acc),
                        "alloc_global": json.dumps(alloc.tolist()),
                        "eval_indices": json.dumps(eval_idx.tolist()),
                        "n_eval_samples": int(len(eval_idx))
                    })

                except Exception as e:
                    print(f"  ERROR: {method_name}, model={model_name}: {e}")

        df = pd.DataFrame(records)

        out_file = os.path.join(
            OUT_DIR,
            f"global_accuracy_results_size_{size_tag}.csv"
        )

        df.to_csv(out_file, index=False)

        print(f"\nSaved: {out_file}")
        print(df[[
            "utility",
            "dataset_size",
            "dataset_train_n",
            "model",
            "method",
            "train_accuracy",
            "test_accuracy",
            "eval_accuracy"
        ]])

if __name__ == "__main__":
    main()